# Conditional copula — simulation study

A single, **configurable** scenario: a Clayton copula whose Kendall's $\tau$ varies
with a covariate $x$. We simulate data, fit `PFNRBicop`, and compare its estimated
conditional **pdf** against the truth (`pyvinecopulib`) across a grid of $(u, v)$
pairs — **with and without** Sinkhorn normalization. Everything below is driven by
the **Configuration** cell.

In [ ]:
import numpy as np
import pandas as pd
import torch
import pyvinecopulib as pv
from dotenv import load_dotenv
from plotnine import (
  aes,
  coord_fixed,
  facet_wrap,
  geom_line,
  geom_point,
  ggplot,
  labs,
  scale_color_cmap,
  theme,
  theme_minimal,
)

from npcc import PFNRBicop

# Load TABPFN_TOKEN (and any other secrets) from the project-root .env.
_ = load_dotenv()

## Configuration

In [ ]:
# --- Scenario: Clayton copula, Kendall's tau linear in the covariate x --------
FAMILY = pv.BicopFamily.clayton
TAU_MIN, TAU_MAX = 0.05, 0.90  # tau(x) ranges linearly over [TAU_MIN, TAU_MAX]
X_MIN, X_MAX = 0.01, 0.99
N_TRAIN = 1000
SEED = 317

# --- Evaluation grid ----------------------------------------------------------
U_LEVELS = (0.1, 0.3, 0.5, 0.7, 0.9)  # one facet per (u, v) pair
V_LEVELS = (0.1, 0.3, 0.5, 0.7, 0.9)
N_X_EVAL = 50  # points along the tau(x) axis

# --- Model --------------------------------------------------------------------
METHOD = "criterion"
SYMMETRIC = True
TRANSFORM = "logit"
PROJECTION_GRID_SIZE = 30  # Sinkhorn projection grid (per axis)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- Sinkhorn sweep (None disables projection) --------------------------------
SINKHORN_VARIANTS = [None, 5]

# pyvinecopulib caps the Clayton parameter at theta <= 28.
CLAYTON_THETA_MAX = 28.0

## Helpers

In [ ]:
def theta_of_tau(tau: np.ndarray) -> np.ndarray:
  """Clayton link: tau = theta / (theta + 2)  =>  theta = 2 * tau / (1 - tau)."""
  tau = np.asarray(tau, dtype=np.float64)
  return 2.0 * tau / (1.0 - tau)


def tau_of_x(x: np.ndarray) -> np.ndarray:
  """Linear Kendall's-tau schedule across the covariate range."""
  x = np.asarray(x, dtype=np.float64)
  frac = (x - X_MIN) / (X_MAX - X_MIN)
  return TAU_MIN + (TAU_MAX - TAU_MIN) * frac


assert theta_of_tau(TAU_MAX) <= CLAYTON_THETA_MAX, (
  f"TAU_MAX={TAU_MAX} gives Clayton theta={theta_of_tau(TAU_MAX):.1f} > "
  f"{CLAYTON_THETA_MAX} (pyvinecopulib bound). Lower TAU_MAX."
)


def sample_scenario(n: int = N_TRAIN, seed: int = SEED) -> pd.DataFrame:
  """Clayton data with pair-specific theta(x) via the inverse conditional CDF."""
  rng = np.random.default_rng(seed)
  x = np.linspace(X_MIN, X_MAX, n, dtype=np.float64)
  theta = theta_of_tau(tau_of_x(x))

  eps = 1e-12
  u = rng.uniform(eps, 1.0 - eps, size=n)
  w = rng.uniform(eps, 1.0 - eps, size=n)

  a = w * (u ** (theta + 1.0))
  term = a ** (-theta / (theta + 1.0)) - u ** (-theta) + 1.0
  v = term ** (-1.0 / theta)
  return pd.DataFrame({"u": u, "v": v, "x": x})

In [ ]:
# Evaluation grids (built once from the configuration above).
UV_PAIRS = [(u, v) for u in U_LEVELS for v in V_LEVELS]
PAIR_LABELS = [f"u={u}, v={v}" for u, v in UV_PAIRS]

X_EVAL = np.linspace(X_MIN, X_MAX, N_X_EVAL)
TAU_EVAL = tau_of_x(X_EVAL)
THETA_EVAL = theta_of_tau(TAU_EVAL)

_U_PAIR = np.asarray([u for u, _ in UV_PAIRS], dtype=np.float64)
_V_PAIR = np.asarray([v for _, v in UV_PAIRS], dtype=np.float64)

# Flat (n_pairs * n_x) grids so the estimator is queried in a single batched call.
_U_FLAT = np.repeat(_U_PAIR[:, None], N_X_EVAL, axis=1).reshape(-1)
_V_FLAT = np.repeat(_V_PAIR[:, None], N_X_EVAL, axis=1).reshape(-1)
_X_FLAT = np.repeat(X_EVAL[None, :], len(UV_PAIRS), axis=0).reshape(-1)


def true_pdf_grid(theta_eval: np.ndarray = THETA_EVAL) -> np.ndarray:
  """True conditional pdf at each (u, v) pair across x, via pyvinecopulib.

  Returns an array of shape ``(n_pairs, n_x)``.
  """
  uv = np.column_stack([_U_PAIR, _V_PAIR])
  cols = [
    pv.Bicop(family=FAMILY, parameters=np.asarray([[theta]])).pdf(uv)
    for theta in theta_eval
  ]
  return np.stack(cols, axis=1)


def estimated_pdf_grid(
  model: PFNRBicop, sinkhorn_iters: int | None
) -> np.ndarray:
  """Estimated conditional pdf via one batched call. Shape ``(n_pairs, n_x)``."""
  flat = model.pdf(_U_FLAT, _V_FLAT, x=_X_FLAT, sinkhorn_iters=sinkhorn_iters)
  return np.asarray(flat).reshape(len(UV_PAIRS), N_X_EVAL)


def _series_label(sinkhorn_iters: int | None) -> str:
  if sinkhorn_iters is None:
    return "Estimated (no Sinkhorn)"
  return f"Estimated (Sinkhorn={sinkhorn_iters})"


def build_pdf_frame(
  model: PFNRBicop, sinkhorn_variants: list[int | None] = SINKHORN_VARIANTS
) -> pd.DataFrame:
  """Tidy long frame with columns ``[pair, tau, value, series]``.

  Written target-agnostically: a cdf / hfunc version only needs a different
  pair of (true, estimated) grid builders.
  """
  grids = {"True": true_pdf_grid()}
  for iters in sinkhorn_variants:
    grids[_series_label(iters)] = estimated_pdf_grid(model, iters)

  records = [
    (label, float(TAU_EVAL[i]), float(grid[j, i]), series)
    for series, grid in grids.items()
    for j, label in enumerate(PAIR_LABELS)
    for i in range(N_X_EVAL)
  ]
  frame = pd.DataFrame(records, columns=["pair", "tau", "value", "series"])
  frame["pair"] = pd.Categorical(frame["pair"], categories=PAIR_LABELS)
  series_order = ["True"] + [_series_label(it) for it in sinkhorn_variants]
  frame["series"] = pd.Categorical(frame["series"], categories=series_order)
  return frame


def facet_plot(frame: pd.DataFrame, ylab: str, title: str) -> ggplot:
  """Faceted estimated-vs-true line plot, one panel per (u, v) pair."""
  return (
    ggplot(frame, aes("tau", "value", color="series", linetype="series"))
    + geom_line(size=0.8)
    + facet_wrap("pair", ncol=len(U_LEVELS), scales="free_y")
    + labs(x=r"Kendall's $\tau(x)$", y=ylab, color="", linetype="", title=title)
    + theme_minimal()
    + theme(figure_size=(16, 16), legend_position="top")
  )

## Scenario data

In [ ]:
df = sample_scenario()
df.head()

In [ ]:
(
  ggplot(df, aes("u", "v", color="x"))
  + geom_point(size=1.2, alpha=0.6)
  + scale_color_cmap(name="viridis")
  + coord_fixed(ratio=1, xlim=(0, 1), ylim=(0, 1))
  + labs(
    x="u",
    y="v",
    color="x",
    title="Simulated Clayton data with covariate-varying dependence",
  )
  + theme_minimal()
)

## Fit `PFNRBicop`

In [ ]:
model = PFNRBicop(
  method=METHOD,
  symmetric=SYMMETRIC,
  transform=TRANSFORM,
  projection_grid_size=PROJECTION_GRID_SIZE,
  device=DEVICE,
)
model.fit(df["u"].to_numpy(), df["v"].to_numpy(), df["x"].to_numpy())
print("fitted on", model._device)

## Conditional pdf — estimated vs true, with/without Sinkhorn

Each panel fixes a $(u, v)$ pair and sweeps the conditional density along
Kendall's $\tau(x)$. The model is fit once; the Sinkhorn variants come from the
per-call `sinkhorn_iters` override on `pdf`.

In [ ]:
pdf_frame = build_pdf_frame(model)
facet_plot(
  pdf_frame,
  ylab=r"$\hat{c}(u, v \mid x)$",
  title="Conditional copula pdf across (u, v) pairs",
)